# ENGR 1451/2451 — Weekly Assignment 10
**10 points**

This assignment focuses on three goals:
(i) interpreting Type I and Type II errors, sensitivity, and positive predictive value using **real biopsy-image data**;
(ii) investigating how significance cutoffs and sample size affect power, using measured features from the same dataset; and
(iii) comparing independent groups and paired observations with $t$-tests.


**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE10-1. Real data: diagnostic flags and base rates (3 pts)

**Dataset:** `./data/LE10_wisconsin_tumors.csv` contains **569 real, labeled biopsy-image observations** from the Wisconsin Diagnostic Breast Cancer dataset. Each row represents a breast mass, with a known diagnosis (`benign` or `malignant`) and measurements extracted from images of cell nuclei. You will use the feature `mean_radius`.

For this classroom exercise, suppose a simple imaging rule **flags** a case when `mean_radius >= 14.0`. A flag means **the rule predicts malignant**; a non-flag means it predicts benign. The threshold is chosen for teaching, **not** as a validated medical screening rule.

The data come from a selected group of diagnostic cases, **not a random population-screening sample**. We will therefore distinguish (1) rates calculated from these real observations from (2) projections under a *hypothetical* population prevalence.

**Source:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic), Wolberg, Mangasarian, Street & Street (1993), DOI: 10.24432/C5DW2B (CC BY 4.0). This teaching CSV contains original real observations but only two of the measured features.

### A. Build the screening table and examine prevalence (2 pts)

Using Python:

1. Load the CSV with `pd.read_csv()` and show the number of cases in each diagnosis category.
2. Using the fixed threshold above, count **true positives (TP)**, **false negatives (FN)**, **false positives (FP)**, and **true negatives (TN)**. Treat *malignant* as the positive class.
3. Display a $2\times 2$ `pd.DataFrame` with **Flagged / Not flagged** as rows and **Malignant / Benign** as columns.
4. Calculate the observed **sensitivity**, **specificity**, **positive predictive value (PPV)**, and **accuracy**:

$$
\mathrm{Sensitivity}=\frac{TP}{TP+FN},\quad
\mathrm{Specificity}=\frac{TN}{TN+FP},\quad
\mathrm{PPV}=\frac{TP}{TP+FP},\quad
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}.
$$

5. **Base-rate thought experiment:** Suppose this rule is applied to **200,000** future cases in a hypothetical population where **0.5%** are malignant. Assume the sensitivity and specificity found in this CSV would transfer unchanged. Compute the *expected* TP, FN, FP, and TN, then the projected PPV. Expected counts need not be whole numbers; round only when displaying them. Compare the projected PPV with the PPV from the observed dataset.

Report percentages to **one decimal place**.


In [1]:
%pip install --upgrade "numpy>=1.26.4,<2.7"
%pip install --upgrade pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
print(os.getcwd())
print(os.listdir('.'))
print(os.listdir('data'))

/ihome/engr1451-2026f/mda90/26f-engr1451-2451/1-assignments/LE10
['.ipynb_checkpoints', 'LE10.ipynb', 'data']
['battery_cooling.csv', 'wisconsin_tumors.csv']


In [10]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)

cases = pd.read_csv("./data/LE10_wisconsin_tumors.csv")
display(cases.head())
print(cases["diagnosis"].value_counts())

threshold = 14.0
# A case is flagged when mean_radius >= threshold.
# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.
# Compute empirical sensitivity, specificity, PPV, and accuracy.
# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.
# Compute the projected PPV (using unrounded expected counts).

# Your code here
threshold = 14.0
actual_positive = cases["diagnosis"].astype(str).str.lower().str.startswith("m")
actual_negative = ~actual_positive
screen_positive = cases["mean_radius"] >= threshold
screen_negative = ~screen_positive
TP = (actual_positive & screen_positive).sum()
FN = (actual_positive & screen_negative).sum()
FP = (actual_negative & screen_positive).sum()
TN = (actual_negative & screen_negative).sum()
screening_table = pd.DataFrame(
    {
        "Malignant": [TP, FN],
        "Benign": [FP, TN]
    },
    index=["Flagged", "Not flagged"]
)

display(screening_table)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
ppv = TP / (TP + FP)
accuracy = (TP + TN) / (TP + TN + FP + FN)

results = pd.Series({
    "sensitivity (%)": sensitivity * 100,
    "specificity (%)": specificity * 100,
    "PPV (%)": ppv * 100,
    "accuracy in percent": accuracy * 100
})

display(results.round(1))

## thought experiment
future_n = 200000
prevalence = 0.005

expected_malignant = future_n * prevalence
expected_benign = future_n * (1 - prevalence)

expected_TP = sensitivity * expected_malignant
expected_FN = (1 - sensitivity) * expected_malignant

expected_TN = specificity * expected_benign
expected_FP = (1 - specificity) * expected_benign

projected_ppv = expected_TP / (expected_TP + expected_FP)

future_results = pd.Series({
    "expected TP": expected_TP,
    "expected FN": expected_FN,
    "expected FP": expected_FP,
    "expected TN": expected_TN,
    "ppv projected %": projected_ppv * 100
})

display(future_results.round(1))

print("observed PPV:", round(ppv * 100, 1), "%")
print("projected PPV:", round(projected_ppv * 100, 1), "%")

,diagnosis,mean_radius,mean_smoothness
0,malignant,17.99,0.1184
1,malignant,20.57,0.0847
2,malignant,19.69,0.1096
3,malignant,11.42,0.1425
4,malignant,20.29,0.1003


diagnosis
benign       357
malignant    212
Name: count, dtype: int64


,Malignant,Benign
Flagged,180,52
Not flagged,32,305


sensitivity (%)        84.9
specificity (%)        85.4
PPV (%)                77.6
accuracy in percent    85.2
dtype: float64

Expected TP             849.1
Expected FN             150.9
Expected FP           28986.0
Expected TN          170014.0
Projected PPV (%)         2.8
dtype: float64

Observed PPV: 77.6 %
Projected PPV: 2.8 %


### B. Interpret the results (1 pt)

Answer in about **3–4 sentences**:

1. For this screening rule, what is a **Type I error** (false positive)? What is a **Type II error** (false negative)?
2. Why does the PPV change so much when the assumed prevalence changes, even if sensitivity and specificity stay fixed? Does a specificity near 85% imply that 85% of flagged cases are malignant?
3. Why should you **not** claim that the observed proportion of malignant cases in this dataset is the prevalence in the general population?

**Answer:**
Type 1 error is a false positive meaning that a benign tumor was flagged as malignant and a false negative is that a malignant tumor was flagged as benign. PPV changes when prevalence changes because it depends on the number of true positives and false positives. If a malignant tumor is shown to be rare, false positives can a big fraction of flagged cases this can happen even if the specificity/sensitivity ; a specificity/sensitivty at 85% doesn't mean that 85% of flagged cases are malignant/benign. A malignant proportion in the data should not be treated as a prevelance in the general population because its not fully representative of it.



## LE10-2. Significance cutoffs, $\beta$, and power (3 pts)

Use the **same real CSV**, but now study another imaging feature, `mean_smoothness`. We are designing a **population-level test based on the mean of $n$ independent biopsies**, *not* screening an individual case. Unlike LE10-1, which uses a fixed threshold to classify individual cases, we now choose cutoffs to control the probability of a Type I error at a specified significance level, $\alpha$.

Estimate the mean and sample SD of `mean_smoothness` separately for **benign** and **malignant** cases in the CSV. For this *planning exercise*, treat those estimated values as fixed benchmarks. Under a **normal approximation for sample means**, model

$$
\bar X\mid H_0 \approx N\!\left(\mu_0,\frac{\sigma_0^2}{n}\right),\qquad
\bar X\mid H_A \approx N\!\left(\mu_1,\frac{\sigma_1^2}{n}\right),
$$

where $H_0$ uses the **benign** reference distribution and the specified $H_A$ uses the **malignant** distribution. A false alarm means rejecting the benign reference when it is true; a miss means failing to reject it when the malignant model is true. **These are approximate design calculations**, not an evaluation of a clinical test or a claim that all biopsy measurements are normally distributed.

Use a **two-sided** decision rule: flag a sample if its mean is below the lower cutoff **or** above the upper cutoff. Compute cutoffs from the *benign* sampling distribution; compute $\beta$ as the probability of falling **between those cutoffs under the malignant model**.

| Plan | Independent biopsies $n$ | Significance level $\alpha$ |
|---|---:|---:|
| A | 10 | 0.05 |
| B | 10 | 0.01 |
| C | 25 | 0.01 |

### A. Calculate error rates and power (2 pts)

Using Python:

1. Extract `mean_smoothness` for benign and malignant cases; calculate each group's sample mean and sample SD using `.std(ddof=1)`.
2. For each plan, define `benign_model` and `malignant_model` with `stats.norm(loc=..., scale=SD/np.sqrt(n))`.
3. Calculate the **lower and upper cutoffs** using `benign_model.ppf([alpha/2, 1-alpha/2])`.
4. Compute $\beta= P(\text{not flagged}\mid H_A)$ using `malignant_model.cdf(upper) - malignant_model.cdf(lower)` and **power** $=1-\beta$.
5. Display a table with plan, $n$, $\alpha$, both cutoffs, $\beta$, and power. Report probabilities to **three decimal places**.

*Use the actual measurements to estimate model parameters; do not simulate additional biopsy records.*


In [11]:
# Use the dataset loaded in LE10-1.
benign = cases.loc[cases["diagnosis"] == "benign", "mean_smoothness"]
malignant = cases.loc[cases["diagnosis"] == "malignant", "mean_smoothness"]

# Calculate mu0, sd0 (benign), and mu1, sd1 (malignant) from the CSV.
# Print both pairs of estimates.

plans = pd.DataFrame({
    "plan": ["A", "B", "C"],
    "n": [10, 10, 25],
    "alpha": [0.05, 0.01, 0.01],
})

# For each plan:
#   - Define benign_model and malignant_model using stats.norm(...).
#   - Calculate cutoffs using benign_model.ppf(...).
#   - Calculate beta using malignant_model.cdf(...).
#   - Calculate power = 1 - beta.
# Display a labeled results table.

# Your code here
# Use the dataset loaded in LE10-1
benign = cases.loc[
    cases["diagnosis"] == "benign",
    "mean_smoothness"
]

malignant = cases.loc[
    cases["diagnosis"] == "malignant",
    "mean_smoothness"
]

mu0 = benign.mean()
sd0 = benign.std(ddof=1)
mu1 = malignant.mean()
sd1 = malignant.std(ddof=1)
print("Benign mean =", mu0)
print("Benign SD =", sd0)
print("Malignant mean =", mu1)
print("Malignant SD =", sd1)
plans = pd.DataFrame({
    "plan": ["A", "B", "C"],
    "n": [10, 10, 25],
    "alpha": [0.05, 0.01, 0.01]
})


results = []

for _, row in plans.iterrows():
    n = int(row["n"])
    alpha = row["alpha"]
    benign_model = stats.norm(
        loc=mu0,
        scale=sd0 / np.sqrt(n)
    )

    malignant_model = stats.norm(
        loc=mu1,
        scale=sd1 / np.sqrt(n)
    )

    lower = benign_model.ppf(alpha / 2)
    upper = benign_model.ppf(1 - alpha / 2)

    beta = (
        malignant_model.cdf(upper)
        - malignant_model.cdf(lower)
    )
    power = 1 - beta

    results.append({
        "plan": row["plan"],
        "n": n,
        "alpha": alpha,
        "lower_cutoff": lower,
        "upper_cutoff": upper,
        "beta": beta,
        "power": power
    })

results = pd.DataFrame(results)
display(results.round({
    "alpha": 3,
    "lower_cutoff": 4,
    "upper_cutoff": 4,
    "beta": 3,
    "power": 3
}))

Benign mean = 0.09247764705882354
Benign SD = 0.013446077330274128
Malignant mean = 0.10289849056603775
Malignant SD = 0.012608235501058329


,plan,n,alpha,lower_cutoff,upper_cutoff,beta,power
0,A,10,0.05,0.0841,0.1008,0.300,0.700
1,B,10,0.01,0.0815,0.1034,0.553,0.447
2,C,25,0.01,0.0856,0.0994,0.083,0.917


### B. Explain the tradeoffs (1 pt)

Answer in about **2–3 sentences**:

1. Compare plans **A and B**: What happens to false alarms ($\alpha$) and missed detections ($\beta$) when the cutoff becomes stricter but $n$ stays the same?
2. Compare plans **B and C**: How does using more **independent biopsies** increase power at the same $\alpha$? Why would this be a study-design consideration rather than a way to improve diagnosis of a *single* patient?

**Answer:**
lowering the alpha will make the test mor constrictive so the amount of false positives/negatives will dcrease but the probability of a missed detection will increase.  for plan b and c increasing the number of biopsies from ten to 25 at the sample alpha value makes the distributions narrower. This increases power. 

## LE10-3. Comparing two mean temperatures (4 pts)

Choose the analysis based on the **study design**, as in w10b. Both parts concern peak battery-module temperature during a standardized load test; **lower is better**.

*The datasets in this problem are simulated classroom examples, not measurements from a published study.*

### A. Independent modules: new vs. standard cooling (2.5 pts)

The file `./data/LE10_battery_cooling.csv` contains peak temperatures ($^\circ$C) for **26 different battery modules**: 12 were randomly assigned to **new** cooling and 14 to **standard** cooling. Each module was measured once.

Let $\Delta=\mu_{\mathrm{new}}-\mu_{\mathrm{standard}}$. We want to know whether the new method **reduces** the population mean peak temperature. Test the boundary null $H_0:\Delta=0$ against $H_A:\Delta<0$, using $\alpha=0.05$.

Assume observations from different modules are independent, group distributions are approximately normal without strong outliers, and population SDs are unknown. Use **Welch's** two-sample $t$-test (do not assume equal population variances).

In [13]:
cooling = pd.read_csv("./data/LE10_battery_cooling.csv")

# The order is new minus standard: a negative difference favors the new method.
new = cooling.loc[cooling["method"] == "new", "peak_temperature_C"].to_numpy()
standard = cooling.loc[cooling["method"] == "standard", "peak_temperature_C"].to_numpy()

display(cooling.head())
print(cooling["method"].value_counts())

,module_id,method,peak_temperature_C
0,N01,new,60.1
1,N02,new,63.0
2,N03,new,59.4
3,N04,new,62.2
4,N05,new,57.8


method
standard    14
new         12
Name: count, dtype: int64


Use Python to:

1. Calculate each group's **sample size, mean, and sample SD** (with `ddof=1`), and the observed difference $\bar x_{\mathrm{new}}-\bar x_{\mathrm{standard}}$.
2. Calculate the estimated SE of the difference:

$$
\widehat{SE}=\sqrt{\frac{s_{\mathrm{new}}^2}{n_{\mathrm{new}}}+\frac{s_{\mathrm{standard}}^2}{n_{\mathrm{standard}}}}.
$$

3. Run `stats.ttest_ind(new, standard, equal_var=False, alternative="less")`. Print the **$t$ statistic, one-sided $P$-value**, and decision at $\alpha=0.05$.
4. For context, run the same Welch test with `alternative="two-sided"` and use its `.confidence_interval(confidence_level=0.95)` method to report a **two-sided 95% confidence interval** for $\Delta$. (You do not need to calculate Welch degrees of freedom manually.)

**Answer:**

In [14]:
# Calculate group summaries (n, sample mean, sample SD).
# Calculate the observed difference and estimated SE.
# Run Welch's one-sided test and print the decision at alpha = 0.05.
# Run a two-sided Welch test and print its 95% confidence interval.

# Your code here
n_new = len(new)
n_standard = len(standard)

mean_new = new.mean()
mean_standard = standard.mean()

sd_new = new.std(ddof=1)
sd_standard = standard.std(ddof=1)
observed_difference = mean_new - mean_standard
estimated_se = np.sqrt(
    sd_new**2 / n_new +
    sd_standard**2 / n_standard
)

summary = pd.DataFrame({
    "group": ["new", "standard"],
    "n": [n_new, n_standard],
    "mean": [mean_new, mean_standard],
    "sample_sd": [sd_new, sd_standard]
})

display(summary)

print("observed difference =", observed_difference)
print("estimated se =", estimated_se)

,group,n,mean,sample_sd
0,new,12,60.9917,1.9071
1,standard,14,64.6571,2.3526


observed difference = -3.665476190476191
estimated se = 0.8357349853183773


### B. Matched modules: before vs. after a firmware change (1.5 pts)

A **separate experiment** measures the *same eight modules* before and after a cooling-control firmware update. The arrays are aligned: `before[i]` and `after[i]` belong to the same module.

1. Calculate the eight within-module differences `after - before` and their mean.
2. Run `stats.ttest_rel(after, before, alternative="less")` to test $H_0:\mu_d=0$ against $H_A:\mu_d<0$, where $d=\text{after}-\text{before}$. Print $t$, the one-sided $P$-value, and the decision at $\alpha=0.05$.
3. In **2–3 sentences total**, interpret the conclusions in parts A and B. Explain **why a paired rather than independent test** is required in part B and whether a statistically significant result by itself establishes the engineering importance of a temperature change.

**Answer:**
Part a is a comparison between two independent froubs. part b shows strong evidence that the update lowered the mean peak temperature. A paired test is required for B because each data in the before is linked to a data in the after measruements. This means that the observations aren't independent. Statistical significane does not cause engineering importance. Size of change in reduction is a factor that must be taken into account as well. 

In [15]:
# Same modules measured twice; each index identifies the same module.
before = np.array([72, 68, 71, 70, 74, 69, 73, 67])
after  = np.array([69, 65, 70, 67, 73, 66, 72, 66])

# Calculate and display within-module differences and their mean.
# Run the paired t-test, report t and P, and make the decision at alpha = 0.05.

# Your code here
before = np.array([72, 68, 71, 70, 74, 69, 73, 67])
after = np.array([69, 65, 70, 67, 73, 66, 72, 66])
differences = after - before
mean_difference = differences.mean()
print("differences:", differences)
print("mean difference:", mean_difference)

# Paired t-test
paired_test = stats.ttest_rel(
    after,
    before,
    alternative="less"
)
t_stat = paired_test.statistic
p_value = paired_test.pvalue
print("t statistic =", t_stat)
print("P-value =", p_value)

if p_value < 0.05:
    print("reject H0 at a = 0.05")
else:
    print("fail to reject H0 at a = 0.05")

differences: [-3 -3 -1 -3 -1 -3 -1 -1]
mean difference: -2.0
t statistic = -5.291502622129181
P-value = 0.0005668915516798224
reject H0 at a = 0.05


## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA10",
  "export_n": 1,
  "session_date": "2026-11-10",
  "messages": [
    {"i": 1, "who": "student", "text": "i am ready to start the homework"},
    {"i": 2, "who": "tutor", "served": ["how_to_use", "a10_preamble", "a10_le10_1"], "text": "The starter cell already loads `cases` and prints the diagnosis counts, so run it first and make sure the `data` folder sits next to your notebook. Then start with two True/False columns: one for whether each case is flagged under the `>= 14.0` rule, and one for whether it is malignant. Which combination of the two makes a true positive?"},
    {"i": 3, "who": "student", "text": "so its asking to coun TPTNFNFP and display it in a 2x2 data frame"},
    {"i": 4, "who": "tutor", "text": "Yes, that's the shape of it: four counts, then a 2×2 `pd.DataFrame` with rows Flagged / Not flagged and columns Malignant / Benign. [trimmed: explained Boolean columns, `&`, `~`, `.sum()` and a labeled DataFrame on an unrelated leak-sensor example; reminded that diagnosis labels are lowercase]"},
    {"i": 5, "who": "student", "text": "<documents>\n\n\n<document index=\"4\" media_type=\"text/plain\"><document_content>/scratch/slurm-44859/ipykernel_1067169/2705724185.py:3: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.24.3)\n  from scipy import stats\n---------------------------------------------------------------------------\nModuleNotFoundError                       Traceback (most recent call last)\nCell In[1], line 3\n      1 import numpy as np\n      2 import pandas as pd\n----> 3 from scipy import stats\n      5 pd.set_option(\"display.precision\", 4)\n      7 cases = pd.read_csv(\"./data/LE10_wisconsin_tumors.csv\")\n\nFile ~/.local/lib/python3.11/site-packages/scipy/__init__.py:131, in __getattr__(name)\n    129 def __getattr__(name):\n    130     if name in submodules:\n--> 131         return _importlib.import_module(f'scipy.{name}')\n    132     else:\n    133         try:\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/importlib/__init__.py:126, in import_module(name, package)\n    124             break\n    125         level += 1\n--> 126 return _bootstrap._gcd_import(name[level:], package, level)\n\nFile ~/.local/lib/python3.11/site-packages/scipy/stats/__init__.py:628\n      1 \"\"\"\n      2 .. _statsrefmanual:\n      3 \n   (...)\n    623 \n    624 \"\"\"  # noqa: E501\n    626 from ._warnings_errors import (ConstantInputWarning, NearConstantInputWarning,\n    627                                DegenerateDataWarning, FitError)\n--> 628 from ._stats_py import *\n    629 from ._variation import variation\n    630 from .distributions import *\n\nFile ~/.local/lib/python3.11/site-packages/scipy/stats/_stats_py.py:39\n     36 import numpy as np\n     37 from numpy import array, asarray, ma\n---> 39 from scipy import sparse\n     40 from scipy.spatial import distance_matrix\n     42 from scipy.optimize import milp, LinearConstraint\n\nFile ~/.local/lib/python3.11/site-packages/scipy/__init__.py:131, in __getattr__(name)\n    129 def __getattr__(name):\n    130     if name in submodules:\n--> 131         return _importlib.import_module(f'scipy.{name}')\n    132     else:\n    133         try:\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/importlib/__init__.py:126, in import_module(name, package)\n    124             break\n    125         level += 1\n--> 126 return _bootstrap._gcd_import(name[level:], package, level)\n\nFile ~/.local/lib/python3.11/site-packages/scipy/sparse/__init__.py:307\n    304 import warnings as _warnings\n    305 import importlib as _importlib\n--> 307 from ._base import *\n    308 from ._csr import *\n    309 from ._csc import *\n\nFile ~/.local/lib/python3.11/site-packages/scipy/sparse/_base.py:8\n      5 import numpy as np\n      6 import operator\n----> 8 from ._sputils import (asmatrix, check_reshape_kwargs, check_shape,\n      9                        get_sum_dtype, isdense, isscalarlike, _todata,\n     10                        matrix, validateaxis, getdtype, is_pydata_spmatrix)\n     11 from scipy._lib._sparse import SparseABC, issparse\n     13 from ._matrix import spmatrix\n\nFile ~/.local/lib/python3.11/site-packages/scipy/sparse/_sputils.py:10\n      8 from math import prod\n      9 import scipy.sparse as sp\n---> 10 from scipy._lib._util import np_long, np_ulong\n     13 __all__ = ['upcast', 'getdtype', 'getdata', 'isscalarlike', 'isintlike',\n     14            'isshape', 'issequence', 'isdense', 'ismatrix', 'get_sum_dtype',\n     15            'broadcast_shapes']\n     17 supported_dtypes = [np.bool_, np.byte, np.ubyte, np.short, np.ushort, np.intc,\n     18                     np.uintc, np_long, np_ulong, np.longlong, np.ulonglong,\n     19                     np.float32, np.float64, np.longdouble,\n     20                     np.complex64, np.complex128, np.clongdouble]\n\nFile ~/.local/lib/python3.11/site-packages/scipy/_lib/_util.py:22\n     19 from scipy._lib._docscrape import FunctionDoc, Parameter\n     20 from scipy._lib._sparse import issparse\n---> 22 from numpy.exceptions import AxisError\n     25 np_long: type\n     26 np_ulong: type\n\nModuleNotFoundError: No module named 'numpy.exceptions'</document_content></document>\n\n\n</documents>\n\nI get this error when I put in my code:", "code": true},
    {"i": 6, "who": "tutor", "text": "[trimmed: read the traceback; SciPy needed a newer NumPy than the kernel had; suggested `%pip install --upgrade \"numpy>=1.26.4,<2.7\"` and a kernel restart, with fallbacks]"},
    {"i": 7, "who": "student", "text": "i did that and now its showing this error: ValueError                                Traceback (most recent call last)\nCell In[3], line 2\n      1 import numpy as np\n----> 2 import pandas as pd\n      3 from scipy import stats\n      5 pd.set_option(\"display.precision\", 4)\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/__init__.py:22\n     19 del _hard_dependencies, _dependency, _missing_dependencies\n     21 # numpy compat\n---> 22 from pandas.compat import is_numpy_dev as _is_numpy_dev  # pyright: ignore # noqa:F401\n     24 try:\n     25     from pandas._libs import hashtable as _hashtable, lib as _lib, tslib as _tslib\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/compat/__init__.py:25\n     17 from pandas.compat._constants import (\n     18     IS64,\n     19     PY39,\n   (...)\n     22     PYPY,\n     23 )\n     24 import pandas.compat.compressors\n---> 25 from pandas.compat.numpy import (\n     26     is_numpy_dev,\n     27     np_version_under1p21,\n     28 )\n     29 from pandas.compat.pyarrow import (\n     30     pa_version_under7p0,\n     31     pa_version_under8p0,\n     32     pa_version_under9p0,\n     33     pa_version_under11p0,\n     34 )\n     37 def set_function_name(f: F, name: str, cls) -> F:\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/compat/numpy/__init__.py:4\n      1 \"\"\" support numpy compatibility across versions \"\"\"\n      2 import numpy as np\n----> 4 from pandas.util.version import Version\n      6 # numpy versioning\n      7 _np_version = np.__version__\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/util/__init__.py:2\n      1 # pyright: reportUnusedImport = false\n----> 2 from pandas.util._decorators import (  # noqa:F401\n      3     Appender,\n      4     Substitution,\n      5     cache_readonly,\n      6 )\n      8 from pandas.core.util.hashing import (  # noqa:F401\n      9     hash_array,\n     10     hash_pandas_object,\n     11 )\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/util/_decorators.py:14\n      6 from typing import (\n      7     Any,\n      8     Callable,\n      9     Mapping,\n     10     cast,\n     11 )\n     12 import warnings\n---> 14 from pandas._libs.properties import cache_readonly\n     15 from pandas._typing import (\n     16     F,\n     17     T,\n     18 )\n     19 from pandas.util._exceptions import find_stack_level\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/_libs/__init__.py:13\n      1 __all__ = [\n      2     \"NaT\",\n      3     \"NaTType\",\n   (...)\n      9     \"Interval\",\n     10 ]\n---> 13 from pandas._libs.interval import Interval\n     14 from pandas._libs.tslibs import (\n     15     NaT,\n     16     NaTType,\n   (...)\n     21     iNaT,\n     22 )\n\nFile /software/rhel9/manual/install/python/ondemand-jupyter-python3.11-2023.09/lib/python3.11/site-packages/pandas/_libs/interval.pyx:1, in init pandas._libs.interval()\n\nValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject", "code": true},
    {"i": 8, "who": "tutor", "text": "[trimmed: explained the pandas/NumPy binary incompatibility after the NumPy upgrade; suggested `%pip install --upgrade pandas` plus a restart, with two fallback routes]"},
    {"i": 9, "who": "student", "text": "now it is saying this: FileNotFoundError                         Traceback (most recent call last)\nCell In[4], line 7\n      3 from scipy import stats\n      5 pd.set_option(\"display.precision\", 4)\n----> 7 cases = pd.read_csv(\"./data/LE10_wisconsin_tumors.csv\")\n      8 display(cases.head())\n      9 print(cases[\"diagnosis\"].value_counts())\n\nFile ~/.local/lib/python3.11/site-packages/pandas/io/parsers/readers.py:1866, in read_csv(filepath_or_buffer, sep, delimiter, header, names, index_col, usecols, dtype, engine, converters, true_values, false_values, skipinitialspace, skiprows, skipfooter, nrows, na_values, keep_default_na, na_filter, skip_blank_lines, parse_dates, date_format, dayfirst, cache_dates, iterator, chunksize, compression, thousands, decimal, lineterminator, quotechar, quoting, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, low_memory, memory_map, float_precision, storage_options, dtype_backend)\n   1853 kwds_defaults = _refine_defaults_read(\n   1854     dialect,\n   1855     delimiter,\n   (...)\n   1862     lineterminator=lineterminator,\n   1863 )\n   1864 kwds.update(kwds_defaults)\n-> 1866 return _read(filepath_or_buffer, kwds)\n\nFile ~/.local/lib/python3.11/site-packages/pandas/io/parsers/readers.py:423, in _read(filepath_or_buffer, kwds)\n    420             return result\n    422 # Create the parser.\n--> 423 parser = TextFileReader(filepath_or_buffer, **kwds)\n    425 if chunksize or iterator:\n    426     return parser\n\nFile ~/.local/lib/python3.11/site-packages/pandas/io/parsers/readers.py:2721, in TextFileReader.__init__(self, f, engine, **kwds)\n   2718     self.options[\"has_index_names\"] = kwds[\"has_index_names\"]\n   2720 self.handles: IOHandles | None = None\n-> 2721 self._engine = self._make_engine(f, self.engine)\n\nFile ~/.local/lib/python3.11/site-packages/pandas/io/parsers/readers.py:2988, in TextFileReader._make_engine(self, f, engine)\n   2983 is_text = engine in (\"python\", \"python-fwf\") or (\n   2984     engine == \"c\"\n   2985     and self.options.get(\"encoding\", \"utf-8\") not in (None, \"utf-8\")\n   2986 )\n   2987 mode = \"r\" if is_text else \"rb\"\n-> 2988 self.handles = get_handle(\n   2989     f,\n   2990     mode,\n   2991     encoding=self.options.get(\"encoding\", None),\n   2992     compression=self.options.get(\"compression\", None),\n   2993     memory_map=self.options.get(\"memory_map\", False),\n   2994     is_text=is_text,\n   2995     errors=self.options.get(\"encoding_errors\", \"strict\"),\n   2996     storage_options=self.options.get(\"storage_options\", None),\n   2997 )\n   2998 assert self.handles is not None\n   2999 f = self.handles.handle\n\nFile ~/.local/lib/python3.11/site-packages/pandas/io/common.py:954, in get_handle(path_or_buf, mode, encoding, compression, memory_map, is_text, errors, storage_options)\n    945         handle = open(\n    946             handle,\n    947             ioargs.mode,\n   (...)\n    950             newline=\"\",\n    951         )\n    952     else:\n    953         # Binary mode\n--> 954         handle = open(handle, ioargs.mode)\n    955     handles.append(handle)\n    957 # Convert BytesIO or file objects passed with an encoding\n\nFileNotFoundError: [Errno 2] No such file or directory: './data/LE10_wisconsin_tumors.csv'", "code": true},
    {"i": 10, "who": "tutor", "text": "[trimmed: explained how the relative path is resolved and gave an `os.getcwd()` / `os.listdir` cell to check the folder and file names]"},
    {"i": 11, "who": "student", "text": "this is what it output: ihome/engr1451-2026f/mda90/26f-engr1451-2451/1-assignments/LE10\n['.ipynb_checkpoints', 'LE10.ipynb', 'data']\n['battery_cooling.csv', 'wisconsin_tumors.csv']"},
    {"i": 12, "who": "tutor", "text": "[trimmed: the files lack the `LE10_` prefix the notebook expects; advised renaming both files via the Jupyter file browser or `os.rename`, run once]"},
    {"i": 13, "who": "student", "text": "can i just delete the leo10"},
    {"i": 14, "who": "tutor", "text": "[trimmed: advised renaming the files rather than editing the paths, so the notebook matches the course file names; warned not to delete `LE10.ipynb`]"},
    {"i": 15, "who": "student", "text": "my code finally works!"},
    {"i": 16, "who": "tutor", "text": "[trimmed: congratulated the student; noted the two diagnosis counts should sum to 569; invited a first attempt at the flagged and malignant True/False columns]"},
    {"i": 17, "who": "student", "text": "[uploaded image: cases.head(), diagnosis counts (benign 357, malignant 212), a 2x2 table (Flagged: 180 malignant, 52 benign; Not flagged: 32 malignant, 305 benign), and rates sensitivity 84.9, specificity 85.4, PPV 77.6, accuracy 85.2]\nthis is what i got:", "code": true},
    {"i": 18, "who": "tutor", "text": "[trimmed: verified the four counts against an independent reference calculation and confirmed row/column labels and rates; noted a cosmetic label inconsistency; introduced the projection idea on an unrelated factory defect-detector example]"},
    {"i": 19, "who": "student", "text": "do I have to write code for the thought experiment?"},
    {"i": 20, "who": "tutor", "text": "[trimmed: yes, Part A asks for it in Python; listed the order of steps, advised unrounded rates and 0.005 for 0.5%]"},
    {"i": 21, "who": "student", "text": "okay so compute the expected for eah case and the the projected ppv"},
    {"i": 22, "who": "tutor", "text": "[trimmed: confirmed the plan; explained that sensitivity applies to the malignant group and specificity to the benign group, with FP from 1 − specificity]"},
    {"i": 23, "who": "student", "text": "okay sounds good."},
    {"i": 24, "who": "tutor", "text": "[trimmed: encouraged the student to write the cell and paste it for checking]"},
    {"i": 25, "who": "student", "text": "so expected is future * prevalence for malignant and the benign is future * 1-prevalence"},
    {"i": 26, "who": "tutor", "text": "[trimmed: confirmed the idea; pointed out that `future * 1 - prevalence` needs parentheses as `future * (1 - prevalence)`]"},
    {"i": 27, "who": "student", "text": "and the TP and FN  TN and FP uses the sensitivity and specificity"},
    {"i": 28, "who": "tutor", "text": "[trimmed: confirmed which rate multiplies which group; gave sum checks (TP+FN, TN+FP, total 200,000); warned to use unrounded proportions]"},
    {"i": 29, "who": "student", "text": "okay got it"},
    {"i": 30, "who": "tutor", "text": "[trimmed: recapped the plan and invited the student to paste the cell or output]"},
    {"i": 31, "who": "student", "text": "[uploaded image: Observed PPV: 77.6 %, Projected PPV: 2.8 %]\nthis is what I got", "code": true},
    {"i": 32, "who": "tutor", "text": "[trimmed: verified the projected PPV against a reference calculation (match); listed the expected counts the student's cell should show; noted the projected counts are required output; offered LE10-1B]"},
    {"i": 33, "who": "student", "text": "Type 1 error is a false positive meaning that a benign tumor was flagged as malignant and a false negative is that a malignant tumor was flagged as benign. The PPV vhanges when prvelance changes due to it depending on the amount of true positives and false positives. If a malignant tumor is shown to be rare, false positives can a big fraction of flagged cases this can happen even if the specificity/sensitivity doesn't change.  That being said a specificity/sensitivty at 85% doesn't mean that 85% of flagged cases are malignant/benign."},
    {"i": 34, "who": "tutor", "text": "[trimmed: gave feedback by question: false-negative wording should match the rule's flag/no-flag language; asked what denominators specificity and PPV each use; noted question 3 (prevalence) was missing; noted length]"},
    {"i": 35, "who": "student", "text": "Type 1 error is a false positive meaning that a benign tumor was flagged as malignant and a false negative is that a malignant tumor was flagged as benign. PPV changes when prevalence changes because it depends on the number of true positives and false positives. If a malignant tumor is shown to be rare, false positives can a big fraction of flagged cases this can happen even if the specificity/sensitivity ; a specificity/sensitivty at 85% doesn't mean that 85% of flagged cases are malignant/benign. A malignant proportion in the data should not be treated as a prevelance in the general population because its not fully representative of it."},
    {"i": 36, "who": "tutor", "text": "[trimmed: gave feedback by question: false-negative wording still mixes flag meaning; fix broken phrases; the 85% point needs the reason (which group each rate is computed within); the prevalence answer should name how the cases were gathered]"},
    {"i": 37, "who": "student", "text": "i will come back to it later can we move onto the next"},
    {"i": 38, "who": "tutor", "served": ["a10_le10_2"]},
    {"i": 39, "who": "student", "text": "[uploaded image: printed benign mean 0.09247764705882354, benign SD 0.013446077330274128, malignant mean 0.10289849056603775, malignant SD 0.012608235501058329, and a table for plans A, B, C with n, alpha, lower_cutoff, upper_cutoff, beta, power: A 0.0841, 0.1008, 0.300, 0.700; B 0.0815, 0.1034, 0.553, 0.447; C 0.0856, 0.0994, 0.083, 0.917]\nthis is what I got", "code": true},
    {"i": 40, "who": "tutor", "text": "[trimmed: verified estimates and every table row against a reference calculation (match); noted formatting; outlined what to examine for LE10-2B]"},
    {"i": 41, "who": "student", "text": "i will ocme back to it"},
    {"i": 42, "who": "tutor", "text": "[trimmed: summarized status (1A and 2A code done, 1B revisions pending, 2B not drafted, LE10-3 not started); reminded how to resume and to request the record]"},
    {"i": 43, "who": "student", "text": "make my record"}
  ],
  "events": [
    {"i": 4, "part": "10-1A", "type": "explain"},
    {"i": 5, "part": "10-1A", "type": "code"},
    {"i": 7, "part": "10-1A", "type": "code"},
    {"i": 9, "part": "10-1A", "type": "code"},
    {"i": 17, "part": "10-1A", "type": "code"},
    {"i": 22, "part": "10-1A", "type": "explain"},
    {"i": 31, "part": "10-1A", "type": "code"},
    {"i": 32, "part": "10-1A", "type": "part_pass"},
    {"i": 39, "part": "10-2A", "type": "code"},
    {"i": 40, "part": "10-2A", "type": "part_pass"}
  ],
  "parts": [
    {"part": "10-1A", "exchanges": 15, "max_tier": 2, "passed": true},
    {"part": "10-1B", "exchanges": 2, "max_tier": 2, "passed": false},
    {"part": "10-2A", "exchanges": 1, "max_tier": 1, "passed": true}
  ],
  "tutor_assessment": [
    "The longest stall was environment setup in 10-1A: a SciPy/NumPy version mismatch, then a pandas binary incompatibility, then data files saved without the LE10_ prefix. The counting, table, rates and projection then went through without a stall. In 10-1B the student had not yet articulated why specificity differs from PPV (the different denominators) or how the dataset's selection affects prevalence, and deferred the revision.",
    "No unstuck declaration or release-valve request was made in this session, so none looked like a tutoring failure. The student chose to defer 10-1B and 10-2B voluntarily, and 10-3 was not started."
  ]
}

## Before you submit

1. Keep `LE10.ipynb` next to a `data` folder containing **both** `LE10_wisconsin_tumors.csv` and the **original** `LE10_battery_cooling.csv`. This revision provides the real Wisconsin CSV; the battery-cooling CSV from the earlier version of LE10 is still needed for Question 3.
2. **Restart Kernel and Run All Cells** after both CSV files are present and confirm every completed cell runs without errors.
3. Confirm your calculated tables, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.
